# 🎓 Study Planner — Multi-Agent Capstone Project
**Track:** Concierge Agents
**Course:** 5-Day AI Agents: Intensive Vibe Coding Course With Google

## What this agent does
Given a learning goal and a deadline, this system produces a **realistic**
day-by-day study schedule — realistic because it actually checks how many
free hours the student has each day (via a live MCP tool), instead of
guessing.

## The 3 concepts demonstrated
1. **Multi-agent system built with ADK** — a `SequentialAgent` chains two
   `LlmAgent`s: `planner_agent` (drafts a schedule) → `reviewer_agent`
   (corrects it against real availability).
2. **MCP server** — a standalone MCP server (`mcp_server.py`) exposes a
   `get_available_hours` tool. `reviewer_agent` calls it live, over stdio,
   exactly like a real calendar integration would work.
3. **Security / guardrail feature** — a `before_model_callback` on both
   agents inspects the user's request *before* any model call happens, and
   blocks unsafe or unrealistic asks (e.g. "study 24/7 with no sleep", or
   "40 hours a day") with a fixed safe response instead.

## Setup
This notebook needs a **free Gemini API key**:
1. Get one at https://aistudio.google.com/apikey
2. On Kaggle: Add-ons → Secrets → add a secret named `GOOGLE_API_KEY`
3. Run the cells below in order


In [ ]:
# Install dependencies
!pip install -q google-adk mcp


In [ ]:
# %%writefile mcp_server.py
mcp_server_code = r'''
"""
mcp_server.py — MCP server exposing get_available_hours (mock calendar tool)
"""
import asyncio, json
from datetime import date, timedelta
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

MOCK_WEEKLY_FREE_HOURS = {0: 2, 1: 3, 2: 2, 3: 3, 4: 4, 5: 6, 6: 5}  # Mon..Sun

def _get_available_hours(num_days: int) -> dict:
    today = date.today()
    schedule = []
    for i in range(num_days):
        day = today + timedelta(days=i)
        free_hours = MOCK_WEEKLY_FREE_HOURS[day.weekday()]
        schedule.append({"date": day.isoformat(), "weekday": day.strftime("%A"), "available_hours": free_hours})
    return {"schedule": schedule, "total_available_hours": sum(d["available_hours"] for d in schedule)}

server = Server("study-calendar-server")

@server.list_tools()
async def list_tools():
    return [Tool(
        name="get_available_hours",
        description="Returns free study hours per day for the next N days, based on the student calendar.",
        inputSchema={"type": "object", "properties": {"num_days": {"type": "integer"}}, "required": ["num_days"]},
    )]

@server.call_tool()
async def call_tool(name, arguments):
    if name == "get_available_hours":
        result = _get_available_hours(int(arguments.get("num_days", 5)))
        return [TextContent(type="text", text=json.dumps(result))]
    raise ValueError(f"Unknown tool: {name}")

async def _main():
    async with stdio_server() as (r, w):
        await server.run(r, w, server.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(_main())
'''

with open("mcp_server.py", "w") as f:
    f.write(mcp_server_code)

print("mcp_server.py written.")


## Step 1 — Security guardrail
This runs *before* every model call and blocks unsafe/unrealistic requests.

In [ ]:
import re
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.genai import types as genai_types

MAX_REASONABLE_DAILY_HOURS = 16

def study_request_guardrail(callback_context: CallbackContext, llm_request: LlmRequest):
    """Blocks unsafe or unrealistic study requests before the model runs."""
    last_user_text = ""
    for content in reversed(llm_request.contents or []):
        if content.role == "user" and content.parts:
            for part in content.parts:
                if getattr(part, "text", None):
                    last_user_text = part.text
                    break
            if last_user_text:
                break

    lowered = last_user_text.lower()

    unsafe_signals = ["no sleep", "without sleep", "all nighter every night", "24 hours a day", "24/7"]
    if any(s in lowered for s in unsafe_signals):
        return LlmResponse(content=genai_types.Content(role="model", parts=[genai_types.Part(text=(
            "I can't build a plan that skips sleep or runs 24/7 — that's not sustainable or safe. "
            "Tell me your goal and deadline and I'll build a realistic schedule instead."
        ))]))

    for match in re.finditer(r"(\d{1,3})\s*hours?", lowered):
        hours = int(match.group(1))
        if hours > MAX_REASONABLE_DAILY_HOURS:
            return LlmResponse(content=genai_types.Content(role="model", parts=[genai_types.Part(text=(
                f"{hours} hours in a day isn't realistic (max ~{MAX_REASONABLE_DAILY_HOURS} productive study hours). "
                "Please give me a more realistic daily limit."
            ))]))

    return None  # safe -> let the real model call proceed

# quick self-test (no API key needed)
class _Fake: pass
def _req(t):
    return LlmRequest(contents=[genai_types.Content(role="user", parts=[genai_types.Part(text=t)])])

print("blocked (24/7):", study_request_guardrail(_Fake(), _req("study 24/7 with no sleep")) is not None)
print("blocked (40 hrs):", study_request_guardrail(_Fake(), _req("study 40 hours a day")) is not None)
print("blocked (normal):", study_request_guardrail(_Fake(), _req("learn SQL in 5 days")) is not None, "<- should be False")


## Step 2 — Connect to the MCP server as a tool

In [ ]:
import sys
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

calendar_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(command=sys.executable, args=["mcp_server.py"]),
        timeout=10.0,
    ),
    tool_filter=["get_available_hours"],
)
print("MCP toolset ready.")


## Step 3 — Build the multi-agent pipeline (Planner → Reviewer)

In [ ]:
from google.adk.agents import LlmAgent, SequentialAgent

planner_agent = LlmAgent(
    name="planner_agent",
    model="gemini-2.5-flash",
    instruction=(
        "You are a study planner. The user gives a learning goal and a deadline in days. "
        "Draft a DAY-BY-DAY study plan (topic + hours per day) to reach the goal by the deadline. "
        "Don't worry about whether the hours are realistic yet - a reviewer checks that next."
    ),
    before_model_callback=study_request_guardrail,
)

reviewer_agent = LlmAgent(
    name="reviewer_agent",
    model="gemini-2.5-flash",
    instruction=(
        "You receive a draft day-by-day study plan. Call get_available_hours to check how many "
        "hours the student actually has free each day. Rewrite the plan so no day exceeds the "
        "available hours - redistribute topics/hours to later days if needed. "
        "Present the FINAL plan as a table: Date | Weekday | Available Hours | Planned Hours | Topic."
    ),
    tools=[calendar_toolset],
    before_model_callback=study_request_guardrail,
)

root_agent = SequentialAgent(
    name="study_planner_pipeline",
    sub_agents=[planner_agent, reviewer_agent],
    description="planner_agent drafts a schedule, reviewer_agent corrects it against real MCP calendar data.",
)

print("Pipeline ready:", [a.name for a in root_agent.sub_agents])


## Step 4 — Run it
Set your Gemini API key below (Kaggle: pull from Secrets; locally: `os.environ`).

In [ ]:
import os

# --- On Kaggle, uncomment these two lines to pull the key from Secrets ---
# from kaggle_secrets import UserSecretsClient
# os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")

# --- Or set it directly for local testing (do NOT commit a real key) ---
# os.environ["GOOGLE_API_KEY"] = "your-key-here"

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY before running this cell."


In [ ]:
import asyncio
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

async def run_planner(goal: str, days: int, user_id="student_1"):
    runner = InMemoryRunner(agent=root_agent, app_name="study_planner_app")
    session = await runner.session_service.create_session(app_name="study_planner_app", user_id=user_id)
    prompt = f"My goal is: {goal}. My deadline is {days} days from today."
    msg = genai_types.Content(role="user", parts=[genai_types.Part(text=prompt)])

    print(f">>> USER REQUEST: {prompt}\n" + "-"*60)
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=msg):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "text", None):
                    print(f"\n[{event.author}]\n{part.text}")
                if getattr(part, "function_call", None):
                    print(f"\n[{event.author}] -> calling tool: {part.function_call.name}({part.function_call.args})")
                if getattr(part, "function_response", None):
                    print(f"\n[{event.author}] <- tool result: {part.function_response.response}")

await run_planner("Learn enough SQL for a job interview", 5)


## Try the guardrail live
This request should get blocked *before* reaching the model.

In [ ]:
await run_planner("I want to study 40 hours a day", 3)


## Summary

| Concept | Where it happens |
|---|---|
| Multi-agent system (ADK) | `SequentialAgent` chaining `planner_agent` → `reviewer_agent` |
| MCP server | `mcp_server.py` running as a subprocess, called live via `McpToolset` |
| Security guardrail | `study_request_guardrail` as `before_model_callback` on both agents |

**Possible extensions:** add a third agent for motivational check-ins, persist
schedules across sessions, or swap the mock calendar for a real Google
Calendar MCP server.
